In [1]:
import joblib
import os
import pandas as pd

print("--- Step 1: Loading Processed ToN-IoT Matrices ---")

processed_dir = "../data/processed/"

# Load the balanced training data and untouched test data
print("Loading data from disk...")
X_train_smote = joblib.load(os.path.join(processed_dir, "ton_iot_X_train_smote.pkl"))
y_train_smote = joblib.load(os.path.join(processed_dir, "ton_iot_y_train_smote.pkl"))
X_test = joblib.load(os.path.join(processed_dir, "ton_iot_X_test.pkl"))
y_test = joblib.load(os.path.join(processed_dir, "ton_iot_y_test.pkl"))

print("\nData successfully loaded!")
print(f"Training Features (SMOTE): {X_train_smote.shape}")
print(f"Training Labels (SMOTE):   {y_train_smote.shape}")
print(f"Testing Features:          {X_test.shape}")
print(f"Testing Labels:            {y_test.shape}")

--- Step 1: Loading Processed ToN-IoT Matrices ---
Loading data from disk...

Data successfully loaded!
Training Features (SMOTE): (1266114, 28)
Training Labels (SMOTE):   (1266114,)
Testing Features:          (199965, 28)
Testing Labels:            (199965,)


In [2]:
from sklearn.ensemble import RandomForestClassifier
import time

print("--- Step 2: Initializing and Training the Random Forest ---")

# Initialize the Random Forest with 100 trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Training the Random Forest on 1.26 million rows... (Please wait)")
start_time = time.time()

# Fit the model!
rf_model.fit(X_train_smote, y_train_smote)

end_time = time.time()
training_time = round(end_time - start_time, 2)
print(f"\nTraining Complete! Elapsed Time: {training_time} seconds")

--- Step 2: Initializing and Training the Random Forest ---
Training the Random Forest on 1.26 million rows... (Please wait)

Training Complete! Elapsed Time: 43.09 seconds


In [3]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import os
import time

print("--- Step 3: Model Evaluation (Unseen Test Set) ---")

# Predict on the untouched 20% test set
start_inference = time.time()
y_pred_rf = rf_model.predict(X_test)
end_inference = time.time()

print(f"Inference Time for {X_test.shape[0]:,} rows: {round(end_inference - start_inference, 4)} seconds")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_rf) * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Normal (0)', 'Attack (1)'], digits=4))

# Generate the Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)

print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (Correctly identified Normal): {cm[0][0]:,}")
print(f"False Positives (False Alarms):               {cm[0][1]:,}")
print(f"False Negatives (Missed Attacks):             {cm[1][0]:,}")
print(f"True Positives (Correctly identified Attack): {cm[1][1]:,}")

# 4. Save the Random Forest model
models_dir = "../models/"
os.makedirs(models_dir, exist_ok=True)
rf_model_path = os.path.join(models_dir, "rf_model_ton_iot.joblib")
joblib.dump(rf_model, rf_model_path)

print(f"\nRandom Forest model successfully saved to: {rf_model_path}")

--- Step 3: Model Evaluation (Unseen Test Set) ---
Inference Time for 199,965 rows: 0.8566 seconds
Accuracy Score: 99.95%

Classification Report:
              precision    recall  f1-score   support

  Normal (0)     0.9982    0.9994    0.9988     41701
  Attack (1)     0.9998    0.9995    0.9997    158264

    accuracy                         0.9995    199965
   macro avg     0.9990    0.9994    0.9992    199965
weighted avg     0.9995    0.9995    0.9995    199965


Confusion Matrix Breakdown:
True Negatives (Correctly identified Normal): 41,674
False Positives (False Alarms):               27
False Negatives (Missed Attacks):             75
True Positives (Correctly identified Attack): 158,189

Random Forest model successfully saved to: ../models/rf_model_ton_iot.joblib
